In [ ]:
pip install qiskit qiskit-aer qiskit-ibm-runtime pylatexenc

Retrieve and Manage Experiment Results

This notebook provides a practical guide to the objects and methods used to retrieve and manage results from Qiskit Primitives jobs.

**Key Concepts Covered:**
- **Objective 1:** Understanding the `SamplerPubResult` object.
- **Objective 2:** Saving job results to disk and retrieving them.
- **Objective 3 & 4:** Exploring the attributes and methods of the `RuntimeJob` and `BasePrimitiveJob` classes.

All examples are designed to run locally using `AerSimulator`, with the cloud-specific sections clearly marked.

## Setup: Run a Simple Sampler Job

To explore the result objects, we first need to run a job. We'll create a simple Bell circuit and run it with the local `BackendSamplerV2`.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2 as Sampler
import numpy as np
import time
import json

# Helper function to create a Bell state circuit
def create_bell_circuit():
  """Create a Bell state (maximally entangled state) circuit."""
  qc = QuantumCircuit(2)
  qc.h(0)  # Apply Hadamard to qubit 0
  qc.cx(0, 1) # Apply CNOT with control=0, target=1
  return qc

# 1. Create a circuit and backend
circuit = create_bell_circuit()
circuit.measure_all()
local_backend = AerSimulator()

# 2. Instantiate a local Sampler and run a job
sampler = Sampler(backend=local_backend)
job = sampler.run([(circuit,)], shots=1024)

# 3. Get the result object
# For local jobs, this is usually instantaneous.
result = job.result()
print('job id')
print(job.job_id())

job id
e925b3c9-b647-4bf4-9ecc-712e5ab4dc9a


## Understanding `SamplerPubResult`

When you get the result from a V2 Primitives job, you get a `PrimitiveResult` object, which is a list of `PubResult` objects (one for each circuit, or "PUB," you submitted). Let's inspect the `PubResult` for our single job.

### Attributes: `data` and `metadata`

The `PubResult` has two main attributes:
*   `.data`: An object containing the core scientific output (e.g., bitstrings for a Sampler, EVs for an Estimator).
*   `.metadata`: A dictionary containing information about how the job was run (e.g., number of shots).

In [ ]:
# Get the result for the first (and only) PUB
pub_result = result[0]

# Access the .data attribute
# For a Sampler, this is a DataBin object with a 'meas' field for measurements.
bitstring_data = pub_result.data.meas

# Access the .metadata attribute
metadata = pub_result.metadata

print(job.job_id())
print(metadata)

# The data object has its own methods to access the results in different formats
print(bitstring_data.get_bitstrings()[:5])
print(bitstring_data.get_counts())

a367a114-e27a-4607-b890-2cd0b28edec6
{'shots': 1024, 'circuit_metadata': {}}
['00', '11', '00', '00', '00']
{'00': 543, '11': 481}


### Methods: `join_data()`

If you run multiple PUBs (multiple circuits) in a single job, you can use `join_data()` to combine their results.

In [ ]:
# Run a job with two different circuits
circuit = create_bell_circuit()
circuit.measure_all()

circuit2 = create_bell_circuit()
circuit2.x(0) # Add X gate to flip the state
circuit2.measure_all()

job_multi = sampler.run([(circuit,), (circuit2,)], shots=512)
result_multi = job_multi.result()

print('result from first circuit')
print(result_multi[0].data.meas.get_counts())

print('result from second circuit')
print(result_multi[1].data.meas.get_counts())

result from first circuit
{'11': 257, '00': 255}
result from second circuit
{'10': 256, '01': 256}


## Objective 2: Save and Retrieve Jobs

For long-running jobs on real hardware, you won't want to wait in your notebook. You can submit a job, get its ID, and then retrieve the result later. This section also shows how to save results to and load them from your disk.

### Save Results to Disk

You can easily save your `PrimitiveResult` object to a JSON file for later analysis. This is useful for sharing results or for post-processing without having to run the job again.

In [ ]:
# We need the Qiskit Runtime JSON encoder to handle the specific data types
from qiskit_ibm_runtime import RuntimeEncoder

# 'result' is the object we got from job.result() earlier
with open("sampler_result.json", "w") as file:
  json.dump(result, file, cls=RuntimeEncoder)

print('saved')

saved


In [ ]:
# Use the Qiskit Runtime JSON decoder to correctly reconstruct the objects
from qiskit_ibm_runtime import RuntimeDecoder

with open("sampler_result.json", "r") as file:
  loaded_result = json.load(file, cls=RuntimeDecoder)

print({type(loaded_result)})

# You can now access the data just as before
print(loaded_result[0].data.meas.get_counts())

{<class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>}
{'00': 543, '11': 481}


### Retrieve Jobs from a Cloud Service (Optional)

**Note:** This section requires a configured `qiskit-ibm-runtime` account. If you don't have one, these cells will raise an `AccountNotFoundError`, which is expected. The `try...except` blocks will catch this and allow the notebook to continue.

In [ ]:
import datetime


try:
  from qiskit_ibm_runtime import QiskitRuntimeService
  QiskitRuntimeService.save_account(
    token="v599YwyyD31O8-qPR_9ecdHoQjWyfmkdjGifgmP2Folk",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/10a28aa5df454bab96a9b4b215b9974f:e4cd8cb3-96e8-4594-bea6-5129384d172a::",
    overwrite=True,
    set_as_default=True,
   )

  # This line will only work if you have an account saved locally
  service = QiskitRuntimeService()

  three_months_ago = datetime.datetime.now() - datetime.timedelta(days=90)
  jobs_in_last_3_months = service.jobs(created_after=three_months_ago)

  print(f'Found {len(list(jobs_in_last_3_months))} jobs from last 90 days')
  if jobs_in_last_3_months:
    print('showing the first 3:')
    for i, job in enumerate(jobs_in_last_3_months[:3]):
      print(f'{i+1}. job {job.job_id()} - Status: {job.status()}')
except Exception as e:
  print(e)
  print('no account')




Found 4 jobs from last 90 days
showing the first 3:
1. job d8m53h032u0s73fao8og - Status: ERROR
2. job d8m13mjqv2lc73873ko0 - Status: CANCELLED
3. job d8jukjj2d42s73c9gv60 - Status: DONE


### Retrieve a Specific Job by ID

If you know a job's ID, you can retrieve it directly. This is useful when you submit a job, save its ID, and come back later to get the results.

In [ ]:
try:
  from qiskit_ibm_runtime import QiskitRuntimeService
  service = QiskitRuntimeService()

  # Get the most recent successful job for demonstration
  successful_jobs = [j for j in service.jobs(limit=100) if j.status() == "DONE"]

  if successful_jobs:
    successful_jobs = successful_jobs[0]
    job_id = successful_jobs.job_id()
    print(f'successful job : {job_id}')

    # Retrieve the job by its ID
    retrieved_job = service.job(job_id)
    retrieved_result = retrieved_job.result()

    print(f'retreived job {job_id}')
    print(f'result :{retrieved_result}')
  else:
    print('No successful jobs')

except Exception as e:
  print(e)
  print("no account found")

successful job : d8jukjj2d42s73c9gv60
retreived job d8jukjj2d42s73c9gv60
result :PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': False, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


## Objective 3 & 4: The `RuntimeJob` and `BasePrimitiveJob`

When you run a job, you get back a `Job` object. For local `BackendPrimitives`, this is a `PrimitiveJob`. For the runtime service, it's a `RuntimeJob`. Both inherit from `BasePrimitiveJob` and share a common set of methods for managing and querying the job's state.

**Important Note:** Some methods are only available on `RuntimeJob` (cloud jobs), not on `PrimitiveJob` (local jobs). We'll clearly mark which is which.

### Job Management Example

Let's run a job on the local simulator and use the job object's methods to monitor its status. For a local job, this will happen very quickly, but on real hardware, a job can stay in the `QUEUED` and `RUNNING` states for a while.

In [ ]:
from qiskit.circuit.library import EfficientSU2
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.jobstatus import JobStatus


# Create a slightly more complex circuit to give the simulator a moment of work
circuit = EfficientSU2(10, reps=4, entanglement='linear')
circuit.measure_all()
params = np.random.rand(circuit.num_parameters)

# Transpile the circuit
pm = generate_preset_pass_manager(optimization_level=1, backend=local_backend)
isa_circuit = pm.run(circuit)

# Submit the job
job = sampler.run([(isa_circuit, params)])
print(job.job_id())

# --- Polling for Job Status ---
# This loop mimics how you would check on a long-running cloud job.
max_checks = 50
checks = 0
while not job.in_final_state() and checks < max_checks:
  # JobStatus is an Enum with values like QUEUED, RUNNING, DONE, ERROR, CANCELLED
  status = job.status()
  print(status)
  time.sleep(0.1) # Wait a moment before checking again
  checks += 1

final_status = job.status()
print(final_status)

# --- Accessing Results ---
if job.done():
  result = job.result()
  print('success')
  print(result[0].data.meas.get_counts())
else:
  # For local jobs, errors are raised immediately when calling result()
  # Cloud jobs have additional methods like errored() and error_message()
  status = job.status()
  print(status)

/tmp/ipykernel_1620/4241969166.py:7: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  circuit = EfficientSU2(10, reps=4, entanglement='linear')


295409ed-bf32-4042-bd8f-f0c774c429d8
JobStatus.RUNNING
JobStatus.DONE
success
{'0101010011': 1, '0110011011': 3, '1011000000': 24, '1101010101': 10, '1000000000': 76, '1001101100': 2, '1110111100': 6, '0110100000': 4, '0101000000': 4, '0010001100': 1, '0101111100': 6, '0010110100': 1, '0110110011': 4, '0000110010': 2, '1000010101': 4, '0111111100': 13, '0100000000': 29, '1100111100': 2, '0110010111': 3, '1100110011': 9, '1011010100': 8, '1011000011': 1, '0000000000': 11, '1000011100': 2, '1011001100': 22, '0001110101': 3, '0110010001': 1, '0010100000': 6, '0110101110': 2, '0110010000': 5, '0110001000': 3, '0000010110': 1, '0101111111': 1, '1101110011': 2, '1100100000': 13, '0100010000': 20, '1011100110': 3, '0101101000': 1, '0110000000': 18, '0110100011': 1, '1101100010': 4, '0111110101': 2, '1001001100': 16, '1101101010': 2, '1111000000': 3, '1100010110': 1, '1011000110': 1, '1111011100': 1, '1100000001': 2, '1101010110': 1, '0011001100': 2, '1110110010': 1, '0111011100': 11, '1101010

### Key Job Attributes and Methods

The `job` object has many useful methods and attributes. Here are the most important ones:

**Methods Available on Both PrimitiveJob (local) and RuntimeJob (cloud):**
* `job.status()`: Returns the job status (e.g., `JobStatus.RUNNING`).
* `job.in_final_state()`: Returns `True` if the job is done, errored, or cancelled.
* `job.done()`: Returns `True` only if the job finished successfully.
* `job.running()`: Returns `True` if the job is currently executing.
* `job.cancelled()`: Returns `True` if the job was cancelled.
* `job.result()`: **Blocks execution** until the job is in a final state and then returns the `PrimitiveResult` object.
* `job.job_id()`: Returns the unique string ID for the job.
* `job.cancel()`: Attempts to cancel a queued or running job.

**Methods ONLY Available on RuntimeJob (cloud jobs):**
* `job.errored()`: Returns `True` if the job failed (not available on local PrimitiveJob).
* `job.error_message()`: Returns the error message if the job failed.
* `job.backend()`: Returns the backend object (may not work on all job types).
* `job.queue_info()`: Information about queue position.
* `job.queue_position()`: The job's position in the queue.
* `job.logs()`: Retrieve job logs.
* `job.metrics()`: Performance metrics for the job.
* `job.update_tags()`: Update job tags.

**RuntimeJob-Specific Attributes:**
* `job.creation_date`
* `job.tags`
* `job.session_id`
* `job.usage_estimation`

In [ ]:
# --- Demonstration of job methods that work on ALL job types ---

# We can get the job ID
print(job.job_id())

# Check if the job is in a final state (should be True now)
print(job.in_final_state())

# Check specific status flags (these work on all jobs)
print(job.done())
print(job.running())
print(job.cancelled())

# Note: errored() is only available on RuntimeJob
# For local jobs, check status directly:
print(job.status())

295409ed-bf32-4042-bd8f-f0c774c429d8
True
True
False
False
JobStatus.DONE
